In [ ]:
from dotenv import load_dotenv
load_dotenv()

# LLM-Guided C++ Kernel Optimization (LangGraph)

We model optimization as a **closed-loop dynamical system**:

    analyze → propose → validate → compile → benchmark → evaluate → repeat

## Design principles
- LLM explores **code transformations**
- System enforces **correctness + measurement**
- State accumulates **performance knowledge**

This is equivalent to:
- autotuning loops
- nonlinear solver iteration
- control systems with feedback

In [ ]:
import logging
import sys

def setup_logger(name="optimizer", level=logging.INFO):
    logger = logging.getLogger(name)
    logger.setLevel(level)

    if not logger.handlers:
        handler = logging.StreamHandler(sys.stdout)
        formatter = logging.Formatter(
            "[%(asctime)s] [%(levelname)s] [iter=%(iteration)s] %(message)s",
            datefmt="%H:%M:%S"
        )
        handler.setFormatter(formatter)
        logger.addHandler(handler)

    return logger

logger = setup_logger()

In [ ]:
def log(state, level, msg):
    extra = {"iteration": state.get("iteration", -1)}
    logger.log(level, msg, extra=extra)

In [ ]:
from typing import TypedDict, List

class OptState(TypedDict):
    code: str
    best_code: str
    best_time: float
    last_time: float
    iteration: int
    max_iters: int          # ← add this
    history: List[str]
    analysis: str
    compile_ok: bool
    artifacts: list

In [ ]:
from langchain.chat_models import init_chat_model
from langchain.agents import create_agent
from langchain.agents.structured_output import ToolStrategy

model = init_chat_model(
    model="gemini-2.5-flash-lite", 
    model_provider="google_genai",
    # Kwargs passed to the model:
    temperature=0.2
)

from pydantic import BaseModel, Field

class CodeProposal(BaseModel):
    code: str = Field(
        description=(
            "Complete, self-contained C++ program. Must compile with g++ -O3 -march=native. "
            "Preserve original semantics exactly. Include all required headers, a valid main() "
            "function, and deterministic behavior. Apply performance optimizations such as "
            "SIMD pragmas, loop transformations, and memory access improvements without "
            "introducing external dependencies."
        )
    )
    strategy: str = Field(
        description=(
            "Concise explanation of the optimization strategy applied. "
            "Focus on performance aspects such as vectorization, memory locality, "
            "loop restructuring, or instruction-level parallelism. "
            "Avoid generic statements; describe concrete changes."
        )
    )


agent = create_agent(
    model = model, 
    response_format = ToolStrategy(CodeProposal)
)   

## Define graph nodes

In [ ]:
def analyze(state: OptState) -> OptState:
    log(state, logging.INFO, "Analyzing code")
    try:
        prompt = f"""
Analyze the performance characteristics of this C++ kernel.

Focus on:
- vectorization
- memory access pattern
- loop structure

Return concise insights.

Code:
{state["code"]}
"""    
        response = model.invoke(prompt)
        
        state["analysis"] = response.content.strip()
        state["history"].append(f"[{state['iteration']}] analysis")
        log(state, logging.INFO, "Analysis complete")
    except Exception as e:
        log(state, logging.ERROR, f"Analysis failed: {e}")
    
    return state

In [ ]:
from langchain_core.messages import HumanMessage, SystemMessage
import json

def propose(state: OptState) -> OptState:
    log(state, logging.INFO, "Generating candidate")
    try:

        sysprompt = "You are an expert in HPC performance optimization using C++"
        humprompt = f"""
You are optimizing a C++ kernel.

Analysis:
{state["analysis"]}

Constraints:
- Preserve semantics exactly
- Improve performance
- No external dependencies

Code:
{state["code"]}
"""

        response = agent.invoke({
            "messages": [
                SystemMessage(content=sysprompt),
                HumanMessage(content=humprompt)
            ]    
        })
        
        obs = response["structured_response"]
        
        state["code"] = obs.code
        state["history"].append(obs.strategy)
        
    except Exception as e:
        log(state, logging.WARNING, f"LLM failure: {e}")


    return state

In [ ]:
def validate(state: OptState) -> OptState:
    log(state, logging.DEBUG, "Validating code")

    if "#include" not in state["code"]:
        state["compile_ok"] = False
        log(state, logging.WARNING, "Validation failed")
    else:
        state["compile_ok"] = True
        log(state, logging.DEBUG, "Validation passed")

    return state

In [ ]:
import subprocess
def compile_code(state: OptState) -> OptState:
    if not state["compile_ok"]:
        log(state, logging.WARNING, "Skipping compile (invalid code)")
        return state

    log(state, logging.INFO, "Writing test.cpp")

    filename = f"test_iter_{state['iteration']}.cpp"
    with open(filename, "w") as f:
        f.write(state["code"])

    log(state, logging.INFO, "Compiling")

    try:
        subprocess.run(
            ["g++", "-O3", "-march=native", filename, "-o", "test.out"],
            check=True,
            capture_output=True
        )
        log(state, logging.INFO, "Compilation succeeded")
        state["artifacts"].append({
            "iteration": state["iteration"],
            "file": filename,
            "time": state["last_time"]
        })

    except subprocess.CalledProcessError:
        state["compile_ok"] = False
        log(state, logging.ERROR, "Compilation failed")

    return state

In [ ]:
def benchmark(state: OptState) -> OptState:
    if not state["compile_ok"]:
        log(state, logging.WARNING, "Skipping benchmark")
        state["last_time"] = float("inf")
        return state

    log(state, logging.INFO, "Benchmarking")

    try:
        result = subprocess.run(["./test.out"], capture_output=True, text=True)
        elapsed = float(result.stdout.strip())

        state["last_time"] = elapsed
        log(state, logging.INFO, f"Runtime: {elapsed:.6e}")

    except Exception as e:
        state["last_time"] = float("inf")
        log(state, logging.ERROR, f"Benchmark failed: {e}")

    return state

In [ ]:
def evaluate(state: OptState) -> str:
    log(state, logging.INFO, "Evaluating candidate")

    if state["iteration"] >= state["max_iters"]:
        log(state, logging.INFO, "Max iterations reached")
        return "stop"

    if state["last_time"] < state["best_time"]:
        state["best_time"] = state["last_time"]
        state["best_code"] = state["code"]
        log(state, logging.INFO, "New best found")

    else:
        log(state, logging.INFO, "No improvement")

    return "loop"

In [ ]:
def increment(state: OptState) -> OptState:
    state["iteration"] += 1
    log(state, logging.DEBUG, "Increment iteration")
    return state

In [ ]:
from langgraph.graph import StateGraph, END

builder = StateGraph(OptState)

builder.add_node("analyze", analyze)
builder.add_node("propose", propose)
builder.add_node("validate", validate)
builder.add_node("compile", compile_code)
builder.add_node("benchmark", benchmark)
builder.add_node("increment", increment)

builder.set_entry_point("analyze")

builder.add_edge("analyze", "propose")
builder.add_edge("propose", "validate")
builder.add_edge("validate", "compile")
builder.add_edge("compile", "benchmark")

builder.add_conditional_edges(
    "benchmark",
    evaluate,
    {
        "loop": "increment",
        "stop": END
    }
)

builder.add_edge("increment", "analyze")

graph = builder.compile()

In [ ]:
baseline_code = r"""
#include <vector>
#include <iostream>
#include <chrono>

void saxpy(int n, float a, const std::vector<float>& x, std::vector<float>& y) {
    for (int i = 0; i < n; ++i) {
        y[i] = a * x[i] + y[i];
    }
}

int main() {
    int n = 1e7;
    std::vector<float> x(n, 1.0f), y(n, 2.0f);

    auto start = std::chrono::high_resolution_clock::now();
    saxpy(n, 2.0f, x, y);
    auto end = std::chrono::high_resolution_clock::now();

    std::chrono::duration<double> diff = end - start;
    std::cout << diff.count() << std::endl;
}
"""

In [ ]:
result = graph.invoke({
    "code": baseline_code,
    "best_code": baseline_code,
    "best_time": float("inf"),
    "last_time": float("inf"),
    "iteration": 0,
    "history": [],
    "analysis": "",
    "compile_ok": True,
    "artifacts": []
})

print("Best time:", result["best_time"])
print("\nHistory:")
print("\n".join(result["history"]))